# A3 · nb00 — environment & data preparation

**CS4.406 Assignment 3: Benchmarking Approximate Nearest Neighbour Search**

This notebook produces every input the later notebooks consume. It does no benchmarking.

1. fingerprints the machine and pins library versions
2. downloads **SIFT-1M** (128-d, Euclidean) and **GloVe-100** (1.18M, angular) from ann-benchmarks
3. unit-normalises GloVe so squared-L2 ordering equals cosine ordering
4. recomputes the exact top-100 ground truth on the T4, **in float64**, and cross-checks it
   against the neighbours stored in each HDF5 file
5. draws nested 100K / 300K SIFT subsets for the Part 3 scaling experiment and computes their GT
6. mirrors everything to `hf://donbosoc/ire-a3-ann`

### Before you run

- **Settings → Accelerator → GPU T4 x2**
- **Settings → Internet → On**
- **Add-ons → Secrets →** add your Hugging Face write token, labelled `HF_TOKEN`, and attach it

Runtime is about 35–50 minutes, most of it the two downloads and the ~1 GB upload.

### Why the GPU is here

It is *not* speeding up the benchmark. The assignment mandates single-thread,
one-query-at-a-time CPU timing, so every measurement in nb01–nb05 runs on `faiss-cpu`
pinned to one thread. The T4 only computes brute-force ground truth — for the normalised
GloVe vectors, the Part 3 subsets and the Part 4 synthetic sets — turning hours of CPU
work into seconds.

## 1 · Install and import

In [ ]:
import subprocess, sys

def pip(*args):
    print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                         capture_output=True, text=True).stdout[-500:] or "ok")

try:
    import faiss
    print(f"faiss already present: {faiss.__version__}")
except ImportError:
    pip("faiss-cpu")
    import faiss
    print(f"faiss installed: {faiss.__version__}")

for mod, pkg in [("h5py", "h5py"), ("huggingface_hub", "huggingface_hub")]:
    try:
        __import__(mod)
    except ImportError:
        pip(pkg)

import h5py, huggingface_hub
print(f"h5py {h5py.__version__} · huggingface_hub {huggingface_hub.__version__}")

## 2 · The shared core

Every notebook in this project writes an identical copy of `annbench_core.py`, so each one
runs standalone with no cross-notebook imports. It holds the tie-aware recall metric, the
timing harness, the GPU ground-truth routine and the results-table writer.

In [ ]:
%%writefile annbench_core.py
"""
annbench_core.py -- shared core for CS4.406 Assignment 3 (ANN benchmarking).

Every notebook writes an identical copy of this file, so each notebook runs standalone.

Conventions used everywhere in this project:
  * All distances are SQUARED L2, computed on float32 vectors. GloVe is unit-normalised
    first, so squared-L2 order == cosine order. We never rely on the scale of the
    distances stored in the ann-benchmarks HDF5 files (euclidean vs cosine units);
    those files are used for their neighbour IDS only, as an independent cross-check.
  * k = 10 everywhere.
  * Queries 0..4999  = TUNE split (parameter picking).
    Queries 5000..9999 = EVAL split (every number that appears in the report).
  * Timing is single-thread, one query at a time. See bench_single_query.
"""

import json
import os
import platform
import re
import subprocess
import sys
import time
import hashlib

import numpy as np

K = 10
TIE_EPS = 1e-6
TUNE = slice(0, 5000)
EVAL = slice(5000, 10000)


def set_split(n_queries):
    """Re-point the tune/eval halves. The full datasets ship 10,000 queries, but quick mode
    uses fewer, and the halves must move with them so no reported number is ever measured
    on a query that was used for tuning."""
    global TUNE, EVAL
    half = n_queries // 2
    TUNE, EVAL = slice(0, half), slice(half, n_queries)
    return TUNE, EVAL

HF_REPO = "donbosoc/ire-a3-ann"
HF_REPO_TYPE = "dataset"

def _pick_scratch():
    """Scratch for big files: /kaggle/temp does not count against the 20 GB output quota,
    but it is not guaranteed to exist or be writable, so fall back rather than crash.
    A3_SCRATCH overrides it entirely, which is how run_all.sh keeps everything in-tree."""
    for cand in ([os.environ["A3_SCRATCH"]] if os.environ.get("A3_SCRATCH") else []) + [
            "/kaggle/temp/a3", "/kaggle/working/tmp/a3", "/tmp/a3"]:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, ".w")
            with open(probe, "w") as f:
                f.write("x")
            os.remove(probe)
            return cand
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory found")


TMP = _pick_scratch()
OUT = os.environ.get("A3_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                                   else os.path.abspath("./out"))


def ensure_dirs():
    for p in (TMP, f"{TMP}/data", f"{TMP}/gt", f"{TMP}/idx",
              f"{OUT}/results", f"{OUT}/figures", f"{OUT}/logs", f"{OUT}/env"):
        os.makedirs(p, exist_ok=True)


# --------------------------------------------------------------------------------------
# Environment
# --------------------------------------------------------------------------------------

def env_fingerprint(verbose=True):
    """Capture the machine + library versions behind every measurement.

    Kaggle VMs are shared and the CPU model can differ between sessions, so this goes
    into every results row and the report states it rather than claiming a quiet machine.
    """
    info = {}
    try:
        cpuinfo = open("/proc/cpuinfo").read()
        models = re.findall(r"^model name\s*:\s*(.+)$", cpuinfo, re.M)
        info["cpu_model"] = models[0].strip() if models else "unknown"
        info["cpu_logical"] = len(models)
    except Exception:
        info["cpu_model"], info["cpu_logical"] = platform.processor() or "unknown", os.cpu_count()
    try:
        mem_kb = int(re.search(r"MemTotal:\s*(\d+)", open("/proc/meminfo").read()).group(1))
        info["mem_gb"] = round(mem_kb / 1048576, 1)
    except Exception:
        info["mem_gb"] = None

    info["python"] = sys.version.split()[0]
    info["numpy"] = np.__version__
    for mod in ("faiss", "torch", "h5py"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["gpu"] = None
    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        info["nvidia_smi"] = None

    # Stable id for "which machine produced this row".
    key = f"{info['cpu_model']}|{info['cpu_logical']}|{info['mem_gb']}"
    info["machine_id"] = hashlib.sha1(key.encode()).hexdigest()[:8]

    if verbose:
        print(f"machine_id : {info['machine_id']}")
        print(f"cpu        : {info['cpu_model']}  ({info['cpu_logical']} logical)")
        print(f"memory     : {info['mem_gb']} GB")
        print(f"gpu        : {info['gpu']}")
        print(f"versions   : python {info['python']}, numpy {info['numpy']}, "
              f"faiss {info['faiss']}, torch {info['torch']}, h5py {info['h5py']}")
    return info


def set_single_thread():
    """Pin FAISS to one thread. Every benchmark number in this project is single-thread."""
    import faiss
    faiss.omp_set_num_threads(1)
    n = faiss.omp_get_max_threads()
    assert n == 1, f"expected 1 FAISS thread, got {n}"
    return n


# --------------------------------------------------------------------------------------
# Hugging Face
# --------------------------------------------------------------------------------------

_TOKEN_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACEHUB_API_TOKEN",
                "HF", "hf_token", "huggingface"]


def hf_enabled():
    """HF mirroring is a convenience for running across Kaggle sessions, never a
    requirement. Everything this project writes also lands on local disk, so with no token
    (the normal case for someone reproducing the results) the mirror is simply skipped."""
    if os.environ.get("A3_NO_HF"):
        return False
    try:
        get_hf_token()
        return True
    except Exception:
        return False


def get_hf_token():
    """Find the HF token in Kaggle secrets, then the environment. Raises if absent."""
    tried = []
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in _TOKEN_NAMES:
            try:
                tok = client.get_secret(name)
                if tok:
                    print(f"HF token: Kaggle secret '{name}'")
                    return tok.strip()
            except Exception:
                tried.append(f"secret:{name}")
    except Exception:
        tried.append("kaggle_secrets unavailable")
    for name in _TOKEN_NAMES:
        tok = os.environ.get(name)
        if tok:
            print(f"HF token: environment variable '{name}'")
            return tok.strip()
        tried.append(f"env:{name}")
    raise RuntimeError(
        "No Hugging Face token found. Add it under Notebook > Add-ons > Secrets with "
        f"label 'HF_TOKEN' and attach it to this notebook. Tried: {tried}")


def hf_api(token=None):
    from huggingface_hub import HfApi
    return HfApi(token=token or get_hf_token())


def hf_ensure_repo(token=None):
    api = hf_api(token)
    api.create_repo(HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    return api


def hf_upload(local_path, path_in_repo, token=None, msg=None, quiet=False, timeout=420):
    """Upload one file, but never let a network stall hang the session.

    huggingface_hub has no hard wall-clock cap on an upload, and a stalled connection will
    block forever. A 12-hour Kaggle session was lost to exactly that. The work runs on a
    daemon thread with a join timeout: if it overruns we warn and carry on, because the
    local copy of everything we upload is always intact on disk.
    """
    import threading

    if not hf_enabled():
        return False
    size = os.path.getsize(local_path) / 1e6
    if not quiet:
        print(f"  -> hf://{HF_REPO}/{path_in_repo}  ({size:.1f} MB)", flush=True)

    box = {}

    def work():
        try:
            hf_api(token).upload_file(
                path_or_fileobj=local_path, path_in_repo=path_in_repo,
                repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                commit_message=msg or f"add {path_in_repo}")
            box["ok"] = True
        except Exception as exc:
            box["err"] = exc

    th = threading.Thread(target=work, daemon=True)
    th.start()
    # Big files legitimately take longer, so scale the allowance with size.
    th.join(timeout + size * 2)
    if th.is_alive():
        print(f"  ! upload of {path_in_repo} still running after "
              f"{timeout + size * 2:.0f}s -- continuing; the local copy is intact",
              flush=True)
        return False
    if "err" in box:
        print(f"  ! upload of {path_in_repo} failed: {box['err']}", flush=True)
        return False
    return True


def hf_download(path_in_repo, token=None, local_dir=None):
    """Return a local path for a repo file, or None if it is not in the repo."""
    from huggingface_hub import hf_hub_download
    if not hf_enabled():
        return None
    try:
        return hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                               filename=path_in_repo, token=token or get_hf_token(),
                               local_dir=local_dir or f"{TMP}/hf")
    except Exception as e:
        if "404" in str(e) or "EntryNotFound" in type(e).__name__:
            return None
        raise


def fetch(repo_path, token=None, verbose=True):
    """Download one prepared artifact from the HF mirror, erroring clearly if nb00 never ran."""
    p = hf_download(repo_path, token=token, local_dir=f"{TMP}/hf")
    if p is None:
        raise FileNotFoundError(
            f"hf://{HF_REPO}/{repo_path} not found -- run nb00_setup_data first")
    if verbose:
        print(f"  {repo_path:44s} {human(os.path.getsize(p))}")
    return p


def load_local(name, with_base=True, with_stored=False):
    """Load what prepare_data.py wrote to local disk (the path run_all.sh uses)."""
    d = {}
    if with_base:
        d["base"] = np.load(f"{TMP}/data/{name}_base.npy")
    d["query"] = np.load(f"{TMP}/data/{name}_query.npy")
    d["gt_ids"] = np.load(f"{TMP}/gt/{name}_top100_ids.npy")
    d["gt_dists"] = np.load(f"{TMP}/gt/{name}_top100_dists.npy")
    if with_stored:
        d["stored"] = np.load(f"{TMP}/data/{name}_stored_neighbors.npy")
    return d


def load_prepared(name, token=None, with_base=True, with_stored=False, verbose=True):
    """Load a dataset prepared by nb00: base, all 10k queries, and our exact top-100 GT.

    Use d["query"][TUNE] for parameter picking and d["query"][EVAL] for reported numbers.
    """
    d = {}
    if with_base:
        d["base"] = np.load(fetch(f"data/{name}_base.npy", token, verbose))
    d["query"] = np.load(fetch(f"data/{name}_query.npy", token, verbose))
    d["gt_ids"] = np.load(fetch(f"gt/{name}_top100_ids.npy", token, verbose))
    d["gt_dists"] = np.load(fetch(f"gt/{name}_top100_dists.npy", token, verbose))
    if with_stored:
        d["stored"] = np.load(fetch(f"data/{name}_stored_neighbors.npy", token, verbose))
    return d


# --------------------------------------------------------------------------------------
# Exact ground truth on the GPU
# --------------------------------------------------------------------------------------

def exact_gt_gpu(base, queries, k=100, chunk=None, mem_budget_gb=None,
                 refine=True, verbose=True):
    """Brute-force top-k by squared L2, selected on the GPU, then refined exactly.

    The GPU pass uses the (|q|^2 + |b|^2 - 2 q.b) expansion in float32, which is fast but
    loses a few ULPs. Selection errors from that can only happen between points that are
    equal to within those ULPs -- i.e. ties, which our recall metric forgives anyway. The
    returned DISTANCES are then recomputed by direct subtraction in float64, so the
    threshold that recall@10 depends on is exact.

    The distance block is (nq, chunk) float32, which at 10k queries is 400 MB per 10k
    base vectors, so a fixed chunk would OOM a 16 GB T4 outright. `chunk` is derived from
    the free VRAM and the query count unless given explicitly, and the block is written
    into ONE preallocated buffer with in-place ops -- writing the expression out as
    `q_sq + b_sq - 2*q@b.T` would allocate three such blocks per iteration instead.

    Returns (ids int32 (nq,k), dists float64 (nq,k)) sorted ascending.
    """
    import torch
    assert base.dtype == np.float32 and queries.dtype == np.float32
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n, d = base.shape
    nq = queries.shape[0]
    k = min(k, n)

    # torch.topk allocates a workspace about the size of its input, so the real peak is
    # ~2x the distance block. Budget a fifth of free VRAM to leave headroom for that.
    if mem_budget_gb is None:
        if dev == "cuda":
            torch.cuda.empty_cache()
            mem_budget_gb = max(0.25, 0.20 * torch.cuda.mem_get_info()[0] / (1 << 30))
        else:
            mem_budget_gb = 2.0
    auto = max(1024, int(mem_budget_gb * (1 << 30) / (nq * 4)))
    chunk = min(chunk, auto) if chunk else auto
    chunk = min(chunk, n)

    q = torch.from_numpy(np.ascontiguousarray(queries).copy()).to(dev)
    q_sq = (q * q).sum(1, keepdim=True)
    buf = torch.empty((nq, chunk), device=dev, dtype=torch.float32)

    best_d = best_i = None
    t0 = time.perf_counter()
    if verbose:
        print(f"    gt: {n:,} x {nq:,} in blocks of {chunk:,} "
              f"(~{nq * chunk * 4 / 1e9:.2f} GB/block) on {dev}")
    for s in range(0, n, chunk):
        blk = torch.from_numpy(np.ascontiguousarray(base[s:s + chunk]).copy()).to(dev)
        m = blk.shape[0]
        dist = buf[:, :m]
        torch.mm(q, blk.T, out=dist)
        dist.mul_(-2.0).add_(q_sq).add_((blk * blk).sum(1)[None, :]).clamp_(min=0)
        kk = min(k, m)
        dv, di = torch.topk(dist, kk, dim=1, largest=False)
        di = di + s
        if best_d is None:
            best_d, best_i = dv.clone(), di
        else:
            cd, ci = torch.cat([best_d, dv], 1), torch.cat([best_i, di], 1)
            best_d, pos = torch.topk(cd, k, dim=1, largest=False)
            best_i = torch.gather(ci, 1, pos)
            del cd, ci, pos
        del blk, dv, di
        if verbose:
            print(f"\r    gt: {min(s + chunk, n):,}/{n:,} vectors", end="", flush=True)
    if verbose:
        print(f"\r    gt: {n:,}/{n:,} vectors in {time.perf_counter() - t0:.1f}s on {dev}")

    ids = best_i.cpu().numpy().astype(np.int32)
    approx = best_d.cpu().numpy().astype(np.float64)
    del q, q_sq, best_i, best_d, buf
    if dev == "cuda":
        torch.cuda.empty_cache()
    if not refine:
        return ids, approx

    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    ids = np.take_along_axis(ids, order, axis=1)
    dists = np.take_along_axis(dists, order, axis=1)
    return ids, dists


def exact_gt(base, queries, k=100, verbose=True):
    """Exact top-k, on the GPU when there is one and through FAISS when there is not.

    The GPU path is far faster for the big ground-truth jobs, but torch is not a hard
    dependency: FAISS's own flat index gives the same ids, and in both cases the returned
    distances are recomputed exactly in float64 so the recall threshold is not approximate.
    """
    try:
        import torch
        if torch.cuda.is_available():
            return exact_gt_gpu(base, queries, k=k, verbose=verbose)
    except ImportError:
        pass

    import faiss
    if verbose:
        print(f"    gt: {len(base):,} x {len(queries):,} via faiss (no GPU)", flush=True)
    ix = faiss.IndexFlatL2(base.shape[1])
    ix.add(base)
    _, ids = ix.search(np.ascontiguousarray(queries), min(k, len(base)))
    ids = ids.astype(np.int32)
    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    return np.take_along_axis(ids, order, axis=1), np.take_along_axis(dists, order, axis=1)


def exact_dists_for_ids(base, queries, ids, block=512):
    """Exact squared-L2 distance from each query to each of its listed ids, in float64.

    Direct subtraction (no q^2+b^2-2qb expansion), so there is no cancellation error.
    ids == -1 (index returned fewer than k) yields +inf. Chunked over queries to bound
    the (block, k, d) temporary.
    """
    nq, kk = ids.shape
    out = np.empty((nq, kk), dtype=np.float64)
    for s in range(0, nq, block):
        e = min(s + block, nq)
        idx = ids[s:e]
        valid = idx >= 0
        vecs = base[np.where(valid, idx, 0)].astype(np.float64)   # (b, k, d)
        diff = vecs - queries[s:e].astype(np.float64)[:, None, :]
        out[s:e] = np.where(valid, np.einsum("bkd,bkd->bk", diff, diff), np.inf)
    return out


# --------------------------------------------------------------------------------------
# Recall@10, tie-aware  (Part 1 asks for this specifically)
# --------------------------------------------------------------------------------------

def recall_at_k(retrieved_ids, retrieved_true_d, gt_dists, k=K, eps=TIE_EPS):
    """Per-query recall@k that does not punish ties at the k-th neighbour.

    A returned id counts as correct when its EXACT distance is <= the exact distance of
    the true k-th neighbour (plus a tiny slack). If the true 10th and 11th neighbours sit
    at the same distance, returning either one is right, so membership of the stored
    top-10 id list is the wrong test -- distance is the right one.

    Duplicate ids are credited once, so an index that returns the same vector twice
    cannot inflate its score.

    retrieved_ids    : (nq, k') ids from the index; -1 means "fewer than k' results".
    retrieved_true_d : (nq, k') EXACT squared-L2 distances for those ids (inf for -1).
    gt_dists         : (nq, >=k) exact squared-L2 distances of the true neighbours, ascending.
    returns          : (nq,) float64 in [0, 1].
    """
    retrieved_ids = np.asarray(retrieved_ids)
    retrieved_true_d = np.asarray(retrieved_true_d, dtype=np.float64)
    gt_dists = np.asarray(gt_dists, dtype=np.float64)
    if retrieved_ids.ndim == 1:
        retrieved_ids = retrieved_ids[None, :]
        retrieved_true_d = retrieved_true_d[None, :]
        gt_dists = gt_dists[None, :]
    assert retrieved_ids.shape == retrieved_true_d.shape
    assert gt_dists.shape[1] >= k, f"need >= {k} ground-truth distances"

    dk = gt_dists[:, k - 1][:, None]
    thresh = dk * (1.0 + eps) + eps           # relative slack, plus absolute for dk == 0
    hit = (retrieved_ids >= 0) & (retrieved_true_d <= thresh)

    # Count DISTINCT hit ids per row.
    masked = np.where(hit, retrieved_ids, -1)
    masked = np.sort(masked, axis=1)
    dup = np.zeros(masked.shape, dtype=bool)
    dup[:, 1:] = masked[:, 1:] == masked[:, :-1]
    n_hit = ((masked >= 0) & ~dup).sum(1)
    return np.minimum(n_hit, k) / float(k)


def recall_from_ids(retrieved_ids, base, queries, gt_dists, k=K):
    """recall_at_k convenience wrapper that recomputes the exact distances itself.

    Needed for IVF-PQ, whose returned distances are approximations of the real ones and
    must never be fed to the recall metric.
    """
    d = exact_dists_for_ids(base, queries, np.asarray(retrieved_ids))
    return recall_at_k(retrieved_ids, d, gt_dists, k=k)


# --------------------------------------------------------------------------------------
# Timing
# --------------------------------------------------------------------------------------

def bench_single_query(search_fn, queries, repeats=3, warmup=50):
    """Time `search_fn` one query at a time, single thread; report the median run.

    Per-query timers cost ~50-100 ns each against queries of 10^2-10^4 us, i.e. under
    0.1%, so QPS is taken as n / sum(per-query latencies) from the same pass that
    produces the latency percentiles.
    """
    nq = len(queries)
    for i in range(min(warmup, nq)):
        search_fn(queries[i:i + 1])

    runs = []
    for _ in range(repeats):
        lat = np.empty(nq, dtype=np.float64)
        for i in range(nq):
            t0 = time.perf_counter_ns()
            search_fn(queries[i:i + 1])
            lat[i] = time.perf_counter_ns() - t0
        runs.append(lat)

    qps = np.array([nq / (r.sum() / 1e9) for r in runs])
    med = runs[int(np.argsort(qps)[len(qps) // 2])]       # latencies of the median run
    return dict(
        qps_median=float(np.median(qps)),
        qps_runs=[round(float(x), 2) for x in qps],
        qps_spread_pct=round(float(100 * (qps.max() - qps.min()) / np.median(qps)), 2),
        lat_p50_ms=float(np.percentile(med, 50) / 1e6),
        lat_p95_ms=float(np.percentile(med, 95) / 1e6),
        n_queries=int(nq), repeats=int(repeats),
    )


def bench_batch(search_fn, queries, repeats=3, warmup=1):
    """Time one call that passes every query at once (Part 1's batched comparison)."""
    nq = len(queries)
    for _ in range(warmup):
        search_fn(queries)
    secs = []
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        search_fn(queries)
        secs.append((time.perf_counter_ns() - t0) / 1e9)
    secs = np.array(secs)
    return dict(qps_median=float(nq / np.median(secs)),
                qps_runs=[round(float(nq / s), 2) for s in secs],
                wall_s_median=float(np.median(secs)),
                n_queries=int(nq), repeats=int(repeats))


# --------------------------------------------------------------------------------------
# Results table
# --------------------------------------------------------------------------------------

COLUMNS = [
    "run_id", "timestamp", "part", "dataset", "n_vectors", "dim", "metric",
    "index_kind", "build_params", "search_param_name", "search_param_value",
    "recall_at_10", "recall_std", "qps_median", "qps_runs", "qps_spread_pct",
    "lat_p50_ms", "lat_p95_ms", "build_time_s", "train_time_s", "add_time_s",
    "index_size_bytes", "bytes_per_vector", "n_queries", "repeats", "threads",
    "query_split", "machine_id", "cpu_model", "faiss_version", "scanned_frac",
    "rc_at_10", "notes",
]


class ResultsLog:
    """Append-only results table, mirrored to HF after every few rows.

    One row per benchmark run, as the deliverable requires. Resumability comes from this
    file: `done()` lets a restarted notebook skip measurements already in the table,
    which is what makes a killed Kaggle session cheap.
    """

    KEY = ("part", "dataset", "n_vectors", "index_kind", "build_params",
           "search_param_name", "search_param_value")

    def __init__(self, env, token=None, path=f"{OUT}/results/results.csv",
                 repo_path="results/results.csv", flush_every=5, pull=True):
        import pandas as pd
        self.pd, self.env, self.token = pd, env, token
        self.path, self.repo_path, self.flush_every = path, repo_path, flush_every
        self._since_flush = 0
        os.makedirs(os.path.dirname(path), exist_ok=True)

        df = None
        if pull and hf_enabled():
            try:
                remote = hf_download(repo_path, token=token)
                if remote:
                    df = pd.read_csv(remote)
                    print(f"results.csv: pulled {len(df)} existing rows from HF")
            except Exception as e:
                print(f"results.csv: could not pull from HF ({e}); starting fresh")
        if df is None and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"results.csv: {len(df)} existing rows found locally")
        if df is None:
            df = pd.DataFrame(columns=COLUMNS)
            print("results.csv: starting a new table")
        for c in COLUMNS:
            if c not in df.columns:
                df[c] = None
        self.df = df[COLUMNS]
        self.df.to_csv(self.path, index=False)

    @staticmethod
    def _sv(x):
        """Normalise a search-param value for comparison: pandas may read an all-numeric
        column back as float, turning 40 into '40.0' and breaking resume detection."""
        t = str(x)
        if t.endswith(".0") and t[:-2].lstrip("-").isdigit():
            t = t[:-2]
        return t

    @staticmethod
    def _norm(params):
        return json.dumps(params, sort_keys=True, separators=(",", ":")) \
            if isinstance(params, dict) else str(params)

    def done(self, part, dataset, n_vectors, index_kind, build_params,
             search_param_name, search_param_value):
        """True when this exact measurement is already in the table."""
        if len(self.df) == 0:
            return False
        want = (str(part), str(dataset), self._sv(n_vectors), str(index_kind),
                self._norm(build_params), str(search_param_name),
                self._sv(search_param_value))
        have = self.df[list(self.KEY)].astype(str).apply(
            lambda col: col.map(self._sv) if col.name in ("n_vectors", "search_param_value")
            else col)
        return bool((have == self.pd.Series(want, index=list(self.KEY))).all(axis=1).any())

    def add(self, *, part, dataset, n_vectors, dim, metric, index_kind, build_params,
            search_param_name, search_param_value, recall, timing, build_time_s=None,
            train_time_s=None, add_time_s=None, index_size_bytes=None,
            query_split="eval", threads=1, scanned_frac=None, rc_at_10=None,
            notes="", flush=False):
        recall = np.asarray(recall, dtype=np.float64)
        timing = timing or {}
        row = {
            "run_id": f"{int(time.time() * 1000)}-{np.random.randint(1e6):06d}",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "part": part, "dataset": dataset, "n_vectors": int(n_vectors), "dim": int(dim),
            "metric": metric, "index_kind": index_kind,
            "build_params": self._norm(build_params),
            "search_param_name": search_param_name, "search_param_value": search_param_value,
            "recall_at_10": round(float(recall.mean()), 6),
            "recall_std": round(float(recall.std()), 6),
            "qps_median": (round(float(timing["qps_median"]), 3)
                           if timing.get("qps_median") is not None else None),
            "qps_runs": json.dumps(timing.get("qps_runs")),
            "qps_spread_pct": timing.get("qps_spread_pct"),
            "lat_p50_ms": timing.get("lat_p50_ms"),
            "lat_p95_ms": timing.get("lat_p95_ms"),
            "build_time_s": build_time_s, "train_time_s": train_time_s,
            "add_time_s": add_time_s, "index_size_bytes": index_size_bytes,
            "bytes_per_vector": (round(index_size_bytes / n_vectors, 2)
                                 if index_size_bytes else None),
            "n_queries": timing.get("n_queries"), "repeats": timing.get("repeats"),
            "threads": int(threads), "query_split": query_split,
            "machine_id": self.env.get("machine_id"), "cpu_model": self.env.get("cpu_model"),
            "faiss_version": self.env.get("faiss"), "scanned_frac": scanned_frac,
            "rc_at_10": rc_at_10, "notes": notes,
        }
        # .loc[] append rather than concat: concat against a frame holding all-NA columns
        # raises a FutureWarning on every single row, which floods the notebook log.
        self.df.loc[len(self.df)] = self.pd.Series(row)
        self.df.to_csv(self.path, index=False)
        self._since_flush += 1
        if flush or self._since_flush >= self.flush_every:
            self.flush()
        return row

    def flush(self):
        self._since_flush = 0
        if not hf_enabled():
            return
        t0 = time.time()
        try:
            hf_upload(self.path, self.repo_path, token=self.token,
                      msg=f"results.csv: {len(self.df)} rows", quiet=True)
        except Exception as e:
            print(f"  ! results.csv upload failed ({e}); the local copy is intact")
        dt = time.time() - t0
        if dt > 30:
            print(f"  (results.csv sync took {dt:.0f}s)", flush=True)


# --------------------------------------------------------------------------------------
# Misc
# --------------------------------------------------------------------------------------

def index_size_bytes(index, path=None):
    """Serialise a FAISS index to disk and measure it. Memory footprint ~= file size for
    these index types, and it is the one number we can take reproducibly."""
    import faiss
    path = path or f"{TMP}/idx/_size_probe.faiss"
    faiss.write_index(index, path)
    n = os.path.getsize(path)
    os.remove(path)
    return n


def normalise(x):
    """Unit-normalise rows in place-ish. Zero rows are left as zeros."""
    x = np.ascontiguousarray(x, dtype=np.float32)
    nrm = np.linalg.norm(x, axis=1, keepdims=True)
    nrm[nrm == 0] = 1.0
    return (x / nrm).astype(np.float32)


def relative_contrast(gt_dists, mean_dist_sq, k=1):
    """RC = (mean distance to a random point) / (distance to the k-th neighbour).

    Reported on true (non-squared) distances, the usual convention. Values near 1 mean
    the nearest neighbour is barely nearer than anything else, which is exactly when
    graph and cell indexes stop working.
    """
    dk = np.sqrt(np.asarray(gt_dists, dtype=np.float64)[:, k - 1])
    dk = np.where(dk == 0, np.finfo(np.float64).tiny, dk)
    return np.sqrt(np.asarray(mean_dist_sq, dtype=np.float64)) / dk


def mean_sq_dist_to_random(base, queries, n_sample=100_000, seed=0):
    """Mean squared distance from each query to a random sample of the base."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(base), size=min(n_sample, len(base)), replace=False)
    sample = base[np.sort(idx)].astype(np.float32)
    s_sq = (sample * sample).sum(1)
    out = np.empty(len(queries), dtype=np.float64)
    for s in range(0, len(queries), 512):
        e = min(s + 512, len(queries))
        q = queries[s:e].astype(np.float32)
        d = (q * q).sum(1)[:, None] + s_sq[None, :] - 2.0 * (q @ sample.T)
        out[s:e] = np.maximum(d, 0).mean(1)
    return out


def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

In [ ]:
%%writefile test_recall.py
"""Unit tests for the tie-aware recall@10 metric (Part 1).

Run: pytest -q test_recall.py      (or: python test_recall.py)
"""

import numpy as np
import pytest

from annbench_core import (K, exact_dists_for_ids, recall_at_k, recall_from_ids)


# ---------------------------------------------------------------------------
# A tiny hand-built dataset where we know every distance exactly.
# 1-D points at 0,1,2,...  query at 0  ->  neighbours are 0,1,2,... in order.
# ---------------------------------------------------------------------------

@pytest.fixture
def line():
    base = np.arange(20, dtype=np.float32).reshape(-1, 1)
    queries = np.zeros((1, 1), dtype=np.float32)
    gt_ids = np.arange(K, dtype=np.int32)[None, :]
    gt_dists = exact_dists_for_ids(base, queries, gt_ids)
    return base, queries, gt_ids, gt_dists


def test_exact_result_is_one(line):
    base, queries, gt_ids, gt_dists = line
    assert recall_from_ids(gt_ids, base, queries, gt_dists)[0] == 1.0


def test_permuted_top10_is_one(line):
    """Recall@10 is a set measure: order within the top 10 must not matter."""
    base, queries, gt_ids, gt_dists = line
    perm = gt_ids[:, ::-1].copy()
    assert recall_from_ids(perm, base, queries, gt_dists)[0] == 1.0


def test_one_genuine_miss_is_point_nine(line):
    base, queries, gt_ids, gt_dists = line
    got = gt_ids.copy()
    got[0, 9] = 15                      # true distance 225 >> d10 = 81
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.9)


def test_all_wrong_is_zero(line):
    base, queries, _, gt_dists = line
    got = np.arange(10, 20, dtype=np.int32)[None, :]
    assert recall_from_ids(got, base, queries, gt_dists)[0] == 0.0


def test_tie_at_tenth_neighbour_is_not_a_mistake():
    """THE case the assignment singles out.

    Points at +-1..+-5 give five exact distance ties. The true 10th neighbour is at
    distance 25, and so is the 11th: both +5 and -5 sit there. An index that returns -5
    where the stored list says +5 has made no error, and must still score 1.0.
    """
    base = np.array([[0.], [1.], [-1.], [2.], [-2.], [3.], [-3.], [4.], [-4.],
                     [5.], [-5.]], dtype=np.float32)
    queries = np.zeros((1, 1), dtype=np.float32)
    stored = np.arange(K, dtype=np.int32)[None, :]          # ids 0..9, i.e. +5 not -5
    gt_dists = exact_dists_for_ids(base, queries, stored)
    assert gt_dists[0, K - 1] == 25.0

    swapped = stored.copy()
    swapped[0, 9] = 10                                      # return -5 instead of +5
    assert recall_from_ids(swapped, base, queries, gt_dists)[0] == 1.0, \
        "a tie at the 10th neighbour was wrongly counted as a mistake"

    # And a naive id-set metric would have scored this 0.9 -- confirm the distinction
    # is real rather than vacuous.
    naive = len(set(swapped[0].tolist()) & set(stored[0].tolist())) / K
    assert naive == 0.9


def test_all_distances_identical_is_one():
    """Degenerate but legal: every point is equidistant, so any 10 ids are correct."""
    base = np.ones((50, 3), dtype=np.float32)
    queries = np.zeros((1, 3), dtype=np.float32)
    gt_dists = exact_dists_for_ids(base, queries, np.arange(K, dtype=np.int32)[None, :])
    got = np.arange(40, 50, dtype=np.int32)[None, :]
    assert recall_from_ids(got, base, queries, gt_dists)[0] == 1.0


def test_zero_distance_ground_truth():
    """d10 == 0 (duplicated vectors): the relative slack vanishes, so the absolute
    slack has to carry the comparison. Guards against a divide-by-zero style bug."""
    base = np.zeros((20, 4), dtype=np.float32)
    base[15:] = 7.0
    queries = np.zeros((1, 4), dtype=np.float32)
    gt_dists = exact_dists_for_ids(base, queries, np.arange(K, dtype=np.int32)[None, :])
    assert gt_dists[0, K - 1] == 0.0
    assert recall_from_ids(np.arange(5, 15, dtype=np.int32)[None, :],
                           base, queries, gt_dists)[0] == 1.0
    assert recall_from_ids(np.arange(10, 20, dtype=np.int32)[None, :],
                           base, queries, gt_dists)[0] == pytest.approx(0.5)


def test_duplicate_ids_are_credited_once(line):
    """An index returning the same vector ten times must score 0.1, not 1.0."""
    base, queries, _, gt_dists = line
    got = np.zeros((1, K), dtype=np.int32)
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.1)


def test_missing_results_are_not_hits(line):
    """FAISS pads with -1 when an index finds fewer than k candidates."""
    base, queries, gt_ids, gt_dists = line
    got = gt_ids.copy()
    got[0, 5:] = -1
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.5)


def test_pq_style_approximate_distances_do_not_corrupt_recall(line):
    """IVF-PQ returns approximate distances. recall_from_ids must ignore whatever the
    index reports and recompute from the base vectors."""
    base, queries, gt_ids, gt_dists = line
    lying = np.zeros((1, K), dtype=np.float64)              # index claims every hit is perfect
    wrong_ids = np.arange(10, 20, dtype=np.int32)[None, :]  # ...but they are all wrong
    assert recall_at_k(wrong_ids, lying, gt_dists)[0] == 1.0          # believing the lie
    assert recall_from_ids(wrong_ids, base, queries, gt_dists)[0] == 0.0   # recomputing


def test_matches_bruteforce_on_random_data():
    """Cross-check against an independent, deliberately naive implementation on random
    float data, where exact ties essentially never occur so the two must agree."""
    rng = np.random.default_rng(0)
    base = rng.standard_normal((400, 8), dtype=np.float32)
    queries = rng.standard_normal((25, 8), dtype=np.float32)

    full = ((queries[:, None, :] - base[None, :, :]) ** 2).sum(-1).astype(np.float64)
    order = np.argsort(full, axis=1)
    gt_ids = order[:, :K].astype(np.int32)
    gt_dists = np.take_along_axis(full, order, axis=1)[:, :K]

    got = order[:, rng.permutation(30)[:K]].astype(np.int32)   # a sloppy "index"
    mine = recall_from_ids(got, base, queries, gt_dists)
    naive = np.array([len(set(g.tolist()) & set(t.tolist())) / K
                      for g, t in zip(got, gt_ids)])
    assert np.allclose(mine, naive)


def test_batch_shape_and_mean(line):
    base, queries, gt_ids, gt_dists = line
    q = np.repeat(queries, 4, axis=0)
    gtd = np.repeat(gt_dists, 4, axis=0)
    got = np.repeat(gt_ids, 4, axis=0).copy()
    got[2, 9] = 19
    r = recall_from_ids(got, base, q, gtd)
    assert r.shape == (4,)
    assert r.mean() == pytest.approx((1 + 1 + 0.9 + 1) / 4)


if __name__ == "__main__":
    raise SystemExit(pytest.main([__file__, "-q"]))

## 3 · Machine fingerprint and Hugging Face auth

Kaggle VMs are shared and the CPU model can change between sessions, so the fingerprint
goes into every row of `results.csv` and the report states it plainly rather than claiming a
quiet dedicated machine. **Check that `machine_id` stays the same across nb01–nb05** — if it
changes, cross-dataset QPS comparisons need a re-run or an explicit caveat.

In [ ]:
import os, sys, json, time, gc, hashlib, subprocess
import numpy as np

sys.path.insert(0, os.getcwd())
import annbench_core as C

C.ensure_dirs()
ENV = C.env_fingerprint()

with open(f"{C.OUT}/env/{ENV['machine_id']}.json", "w") as f:
    json.dump(ENV, f, indent=2)

TOKEN = C.get_hf_token()
C.hf_ensure_repo(TOKEN)
print(f"\nHF repo ready: https://huggingface.co/datasets/{C.HF_REPO}")
print(f"scratch: {C.TMP}   output: {C.OUT}")

### Sanity-check the recall metric right away

The full unit-test discussion belongs to Part 1 (nb01), but a broken recall function would
silently poison the ground-truth cross-check below, so the tests run here too.

In [ ]:
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "test_recall.py"],
                     capture_output=True, text=True).stdout[-2000:])

## 4 · Download the datasets

HF mirror first (fast, and immune to upstream going away), ann-benchmarks.com as the
fallback. `wget -c` resumes a partial file, so a dropped connection costs nothing.

In [ ]:
DATASETS = {
    "sift":  dict(url="https://ann-benchmarks.com/sift-128-euclidean.hdf5",
                  file="sift-128-euclidean.hdf5", metric="euclidean", normalise=False),
    "glove": dict(url="https://ann-benchmarks.com/glove-100-angular.hdf5",
                  file="glove-100-angular.hdf5", metric="angular", normalise=True),
}

def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"  cached: {dest} ({C.human(os.path.getsize(dest))})")
        return dest
    print(f"  downloading {url}")
    r = subprocess.run(["wget", "-c", "-q", "--show-progress", "--progress=dot:giga",
                        "-O", dest, url], capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(dest):
        raise RuntimeError(f"download failed: {r.stderr[-800:]}")
    print(f"  got {C.human(os.path.getsize(dest))}")
    return dest

for name, spec in DATASETS.items():
    print(f"[{name}]")
    spec["path"] = fetch(spec["url"], f"{C.TMP}/{spec['file']}")

for name, spec in DATASETS.items():
    with h5py.File(spec["path"], "r") as f:
        print(f"\n[{name}] keys={list(f.keys())}")
        for k in f.keys():
            print(f"   {k:12s} {str(f[k].shape):22s} {f[k].dtype}")
        spec["n"], spec["d"] = f["train"].shape

## 5 · To `.npy`, with GloVe normalised

GloVe is distributed for angular distance. Unit-normalising every vector makes squared-L2
rank-equivalent to cosine, so the whole project can use one metric and one exact baseline —
which is what the assignment asks for. The file's own `distances` are in cosine units and we
deliberately never use their scale; only the neighbour **ids** are used, as an independent
check on our own ground truth.

In [ ]:
def load_split(spec):
    with h5py.File(spec["path"], "r") as f:
        base = np.ascontiguousarray(f["train"][:], dtype=np.float32)
        query = np.ascontiguousarray(f["test"][:], dtype=np.float32)
        stored_nb = np.ascontiguousarray(f["neighbors"][:], dtype=np.int32)
    if spec["normalise"]:
        nz = (np.linalg.norm(base, axis=1) == 0).sum()
        base, query = C.normalise(base), C.normalise(query)
        print(f"  normalised; {nz} zero-norm base vectors left as zeros")
        print(f"  base norms  min {np.linalg.norm(base, axis=1).min():.6f} "
              f"max {np.linalg.norm(base, axis=1).max():.6f}")
    return base, query, stored_nb

PREP = {}
for name, spec in DATASETS.items():
    print(f"[{name}]")
    base, query, stored_nb = load_split(spec)
    print(f"  base {base.shape} {base.dtype} · queries {query.shape} · stored GT {stored_nb.shape}")
    np.save(f"{C.TMP}/data/{name}_base.npy", base)
    np.save(f"{C.TMP}/data/{name}_query.npy", query)
    np.save(f"{C.TMP}/data/{name}_stored_neighbors.npy", stored_nb)
    PREP[name] = dict(n=base.shape[0], d=base.shape[1])
    del base, query, stored_nb
    gc.collect()

print("\nQuery split: tune = queries[0:5000], eval = queries[5000:10000].")
print("Every number in the report comes from the eval half only.")

## 6 · Exact ground truth, and the cross-check that matters

The GPU selects the top 100 by the `|q|²+|b|²−2q·b` expansion in float32 (fast), then the
returned distances are **recomputed by direct subtraction in float64** (exact). Selection
error from the expansion can only mis-order points that are equal to within a few ULPs —
i.e. ties, which the recall metric forgives by construction — while the *threshold* recall@10
depends on ends up exact.

Because the refine step re-sorts the top 100 exactly, the final **top-10 is exact** unless the
float32 pass dropped a true top-10 point out of its top *100*, which would take an error of
order (d100 − d10) — thousands of times larger than the ~1 ULP we actually see.

The check below asks the independent question: does the ground truth we computed agree with
the neighbours shipped in the HDF5 file? Scored with our tie-aware metric, this should be
≈ 1.000. Anything materially below that means our pipeline disagrees with the dataset authors
and nothing downstream is trustworthy.

The block size adapts to free VRAM: a (10000 × chunk) float32 distance block plus `topk`'s
own workspace is what sets the limit, not the dataset size.

In [ ]:
GT_K = 100
gt_report = {}

for name in DATASETS:
    print(f"[{name}]")
    base = np.load(f"{C.TMP}/data/{name}_base.npy")
    query = np.load(f"{C.TMP}/data/{name}_query.npy")

    t0 = time.perf_counter()
    gt_ids, gt_dists = C.exact_gt_gpu(base, query, k=GT_K)
    secs = time.perf_counter() - t0

    np.save(f"{C.TMP}/gt/{name}_top100_ids.npy", gt_ids)
    np.save(f"{C.TMP}/gt/{name}_top100_dists.npy", gt_dists)

    stored_nb = np.load(f"{C.TMP}/data/{name}_stored_neighbors.npy")
    agree_exact = float((stored_nb[:, :10] == gt_ids[:, :10]).all(axis=1).mean())
    rec_stored = C.recall_from_ids(stored_nb[:, :10], base, query, gt_dists)
    rec_self = C.recall_from_ids(gt_ids[:, :10], base, query, gt_dists)

    print(f"  computed in {secs:.1f}s")
    print(f"  d1  mean {np.sqrt(gt_dists[:, 0]).mean():.4f} · "
          f"d10 mean {np.sqrt(gt_dists[:, 9]).mean():.4f}  (true distances)")
    print(f"  queries whose stored top-10 is id-identical to ours : {agree_exact:.4f}")
    print(f"  recall@10 of the FILE's neighbours vs our GT        : {rec_stored.mean():.6f}")
    print(f"  recall@10 of our own GT vs itself (must be 1.0)     : {rec_self.mean():.6f}")
    assert rec_self.mean() == 1.0, "self-recall must be exactly 1.0"
    assert rec_stored.mean() > 0.999, "our ground truth disagrees with the dataset file"

    gt_report[name] = dict(seconds=round(secs, 1), id_identical=agree_exact,
                           recall_file_vs_ours=round(float(rec_stored.mean()), 6),
                           d1_mean=float(np.sqrt(gt_dists[:, 0]).mean()),
                           d10_mean=float(np.sqrt(gt_dists[:, 9]).mean()))
    del base, query, gt_ids, gt_dists, stored_nb
    gc.collect()

print("\nGround truth verified against both dataset files.")

### Relative contrast of each dataset

How much nearer is the 10th neighbour than a random point? This single number explains most
of the recall differences we expect between SIFT and GloVe, so it is worth measuring now
rather than asserting later. It is also the quantity the Part 4 experiment varies on purpose.

In [ ]:
contrast = {}
for name in DATASETS:
    base = np.load(f"{C.TMP}/data/{name}_base.npy")
    query = np.load(f"{C.TMP}/data/{name}_query.npy")
    gt_dists = np.load(f"{C.TMP}/gt/{name}_top100_dists.npy")
    msq = C.mean_sq_dist_to_random(base, query, n_sample=100_000, seed=0)
    rc1 = C.relative_contrast(gt_dists, msq, k=1)
    rc10 = C.relative_contrast(gt_dists, msq, k=10)
    contrast[name] = dict(rc_at_1=float(rc1.mean()), rc_at_10=float(rc10.mean()))
    print(f"[{name}] relative contrast: RC@1 {rc1.mean():.3f} · RC@10 {rc10.mean():.3f}  "
          f"(higher = easier)")
    del base, query, gt_dists
    gc.collect()

## 7 · Nested subsets for the Part 3 scaling experiment

Part 3 re-runs the SIFT benchmark at 100K, 300K and 1M. The subsets are drawn with a fixed
seed and **nested** (100K ⊂ 300K ⊂ 1M), so a difference between sizes is attributable to size
alone and not to having sampled a different region of the space. We store only the chosen
**ids** plus their ground truth — the vectors themselves are recoverable from the 1M base by
fancy-indexing, which keeps ~200 MB off the upload.

Ground-truth ids for a subset are **local** (0..n−1), matching what an index built on that
subset will return.

In [ ]:
SUBSET_SEED = 20261023
SUBSET_SIZES = [100_000, 300_000]

base = np.load(f"{C.TMP}/data/sift_base.npy")
query = np.load(f"{C.TMP}/data/sift_query.npy")
rng = np.random.default_rng(SUBSET_SEED)
perm = rng.permutation(len(base))

subset_report = {}
for n in SUBSET_SIZES:
    tag = f"sift{n // 1000}k"
    ids = np.sort(perm[:n]).astype(np.int32)      # nested: prefixes of one permutation
    np.save(f"{C.TMP}/data/{tag}_ids.npy", ids)
    sub = np.ascontiguousarray(base[ids])
    gt_ids, gt_dists = C.exact_gt_gpu(sub, query, k=GT_K)
    np.save(f"{C.TMP}/gt/{tag}_top100_ids.npy", gt_ids)
    np.save(f"{C.TMP}/gt/{tag}_top100_dists.npy", gt_dists)
    assert gt_ids.max() < n and gt_ids.min() >= 0, "subset GT ids must be local"
    assert C.recall_from_ids(gt_ids[:, :10], sub, query, gt_dists).mean() == 1.0
    full_d10 = np.load(f"{C.TMP}/gt/sift_top100_dists.npy")[:, 9]
    print(f"[{tag}] n={n:,}  d10 mean {np.sqrt(gt_dists[:, 9]).mean():.2f} "
          f"(1M: {np.sqrt(full_d10).mean():.2f})")
    subset_report[tag] = dict(n=n, d10_mean=float(np.sqrt(gt_dists[:, 9]).mean()))
    del sub, gt_ids, gt_dists
    gc.collect()

prev = None
for n in SUBSET_SIZES:
    cur = set(np.load(f"{C.TMP}/data/sift{n // 1000}k_ids.npy").tolist())
    if prev is not None:
        assert prev <= cur, "subsets are not nested"
    prev = cur
print("\nSubsets are nested and their ground truth checks out.")
del base, query
gc.collect()

## 8 · Mirror to Hugging Face

Bases and queries go up as `.npy` so later notebooks start in seconds instead of re-pulling
from ann-benchmarks.com. The raw HDF5 files are *not* mirrored — everything in them that we
use (vectors, queries, stored neighbour ids) is already captured above, so mirroring them
would double the upload for nothing.

In [ ]:
UPLOAD_BASES = True       # ~1 GB; set False to upload only the small artifacts

manifest = {"created": time.strftime("%Y-%m-%dT%H:%M:%S"), "env": ENV,
            "gt_check": gt_report, "relative_contrast": contrast,
            "subsets": subset_report, "subset_seed": SUBSET_SEED,
            "gt_k": GT_K, "query_split": {"tune": [0, 5000], "eval": [5000, 10000]},
            "files": {}}

uploads = []
for name in DATASETS:
    if UPLOAD_BASES:
        uploads.append((f"{C.TMP}/data/{name}_base.npy", f"data/{name}_base.npy"))
    uploads += [(f"{C.TMP}/data/{name}_query.npy", f"data/{name}_query.npy"),
                (f"{C.TMP}/data/{name}_stored_neighbors.npy",
                 f"data/{name}_stored_neighbors.npy"),
                (f"{C.TMP}/gt/{name}_top100_ids.npy", f"gt/{name}_top100_ids.npy"),
                (f"{C.TMP}/gt/{name}_top100_dists.npy", f"gt/{name}_top100_dists.npy")]
for n in SUBSET_SIZES:
    tag = f"sift{n // 1000}k"
    uploads += [(f"{C.TMP}/data/{tag}_ids.npy", f"data/{tag}_ids.npy"),
                (f"{C.TMP}/gt/{tag}_top100_ids.npy", f"gt/{tag}_top100_ids.npy"),
                (f"{C.TMP}/gt/{tag}_top100_dists.npy", f"gt/{tag}_top100_dists.npy")]

for local, remote in uploads:
    a = np.load(local, mmap_mode="r")
    manifest["files"][remote] = dict(shape=list(a.shape), dtype=str(a.dtype),
                                     bytes=os.path.getsize(local))
    del a

t0 = time.perf_counter()
for i, (local, remote) in enumerate(uploads, 1):
    print(f"[{i}/{len(uploads)}]", end=" ")
    C.hf_upload(local, remote, token=TOKEN, msg="nb00: prepared data")
print(f"\nuploaded {len(uploads)} files in {time.perf_counter() - t0:.0f}s")

with open(f"{C.OUT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
C.hf_upload(f"{C.OUT}/manifest.json", "manifest.json", token=TOKEN, msg="nb00: manifest")
C.hf_upload("annbench_core.py", "code/annbench_core.py", token=TOKEN, msg="nb00: core")
C.hf_upload("test_recall.py", "code/test_recall.py", token=TOKEN, msg="nb00: tests")
C.hf_upload(f"{C.OUT}/env/{ENV['machine_id']}.json",
            f"logs/env/{ENV['machine_id']}.json", token=TOKEN, msg="nb00: env")
print(f"\nhttps://huggingface.co/datasets/{C.HF_REPO}/tree/main")

## 9 · Summary — paste this back

In [ ]:
print("=" * 72)
print("nb00 SUMMARY")
print("=" * 72)
print(f"machine_id   : {ENV['machine_id']}")
print(f"cpu          : {ENV['cpu_model']} ({ENV['cpu_logical']} logical), {ENV['mem_gb']} GB RAM")
print(f"gpu          : {ENV['gpu']}")
print(f"versions     : faiss {ENV['faiss']} · torch {ENV['torch']} · numpy {ENV['numpy']} "
      f"· h5py {ENV['h5py']} · python {ENV['python']}")
print()
for name in DATASETS:
    p, g, c = PREP[name], gt_report[name], contrast[name]
    print(f"{name:6s} n={p['n']:>9,}  d={p['d']:>4}  "
          f"GT {g['seconds']:>5.1f}s  file-vs-ours recall {g['recall_file_vs_ours']:.6f}  "
          f"id-identical {g['id_identical']:.4f}")
    print(f"{'':6s} d1 {g['d1_mean']:.4f} · d10 {g['d10_mean']:.4f} · "
          f"RC@1 {c['rc_at_1']:.3f} · RC@10 {c['rc_at_10']:.3f}")
for tag, s in subset_report.items():
    print(f"{tag:6s} n={s['n']:>9,}  d10 mean {s['d10_mean']:.2f}")
print()
print(f"free disk /kaggle/temp : {C.human(__import__('shutil').disk_usage(C.TMP).free)}")
print(f"HF repo                : https://huggingface.co/datasets/{C.HF_REPO}")
print("=" * 72)

### What happens next

`nb01` (Part 1) builds the exact FAISS baseline on top of these artifacts: it runs the recall
unit tests properly, confirms exact search reproduces the stored neighbours, and times exact
search one-query-at-a-time against all-queries-in-one-batch on both datasets.

Paste the summary above back before moving on — the measured GT timings, relative contrasts
and `machine_id` determine the sweep grids and time budget for nb02/nb03.